# Experiment H6: Learning Rate Artifact Check

## Is the 130x Curvature Advantage Just a Learning Rate Artifact?

### Scientific Motivation

Experiment 3.4 demonstrated that curvature-rescaled Muon (k=5) achieved a **~130x improvement** in final
loss over vanilla Muon. On the surface, this looks like powerful evidence that per-step curvature adaptation
is valuable. But a closer inspection of the rescaling statistics revealed a critical red flag:

> **The rescale factor hit its minimum clamp (0.1) in 96.5% of all steps.**

This means the curvature rescaler was, in practice, doing almost nothing adaptive. It was simply multiplying
the orthogonalized gradient by 0.1 on nearly every step. Since the gradient update rule is:

$$W_{t+1} = W_t - \text{lr} \cdot v_t, \quad v_t = \beta v_{t-1} + \text{scale} \cdot G_{\text{orth}}$$

multiplying by `scale = 0.1` is algebraically equivalent to reducing the effective learning rate by 10x:

$$\text{lr}_{\text{effective}} = \text{lr} \times \text{scale} = 0.02 \times 0.1 = 0.002$$

**If vanilla Muon at lr=0.002 matches the rescaled Muon result, then the entire 130x "curvature advantage"
is an artifact of the default learning rate (0.02) being 10x too high.**

### What This Experiment Tests

| Test | Question | Pass Criterion |
|------|----------|----------------|
| **T1** | Is the best vanilla Muon LR near 0.002? | Within factor of 2 of 0.02 x 0.1 |
| **T2** | Does best-LR vanilla Muon match rescaled Muon? | Within 5% of rescaled loss |
| **T3** | Does rescaling at the best LR provide further benefit? | >5% additional improvement |

### Experimental Design

- **Architecture**: 2-layer 4x4 deep linear network (same as 3.4)
- **Training**: 500 steps, momentum=0.9, 10 random seeds
- **Vanilla Muon LR sweep**: {0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1}
- **SGD LR sweep**: {0.0005 ... 0.5} for baseline context
- **Curvature-rescaled Muon**: at lr=0.02 (3.4 reference) and at the best vanilla LR

### Why This Matters

If the 130x is an artifact, it means our curvature rescaling mechanism has no genuine signal -- it is just
accidentally finding a better LR via its clamp. This would invalidate the claim that Muon benefits from
per-step curvature adaptation, and redirect research toward understanding why Muon's default LR is
suboptimal for this architecture. Conversely, if rescaling provides genuine benefit even at the optimal LR,
it validates the curvature adaptation hypothesis and motivates deeper investigation of the rescaling mechanism.

## Imports and Environment

In [ ]:
import numpy as np
import os

print(f"NumPy version: {np.__version__}")
print(f"Random seed strategy: deterministic per-seed initialization for reproducibility")

In [ ]:
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__))
print(f"Script directory: {SCRIPT_DIR}")

## Experimental Configuration

### Architecture and Training Parameters

All parameters are chosen to exactly replicate the Experiment 3.4 setup, ensuring a fair apples-to-apples
comparison. The deep linear network is intentionally small (4x4, 2 layers) to isolate the optimization
dynamics from capacity effects.

**Key design choices:**
- **DIM=4, NUM_LAYERS=2**: Matches 3.4 exactly. Small enough for exhaustive analysis, large enough for
  nontrivial spectral structure (4 singular values per layer).
- **NUM_STEPS=500**: Sufficient for convergence at most LRs. If a method hasn't converged by 500 steps,
  it likely never will (divergence or extremely slow progress).
- **MOMENTUM=0.9**: Standard heavy-ball momentum. Important because the rescaling interacts with
  momentum -- a scale factor of 0.1 on the gradient feeds into the velocity buffer, so the effective
  LR reduction is sustained across steps.
- **SCALE_MIN=0.1, SCALE_MAX=10.0**: The clamp range for curvature rescaling. The fact that 96.5% of
  steps hit SCALE_MIN is the entire reason this experiment exists.
- **NUM_SEEDS=10**: Enough for robust statistics while keeping runtime manageable.

In [ ]:
DIM = 4
NUM_LAYERS = 2
NUM_STEPS = 500
MOMENTUM = 0.9
GAMMA = 1.0
SCALE_MIN = 0.1
SCALE_MAX = 10.0
NUM_SEEDS = 10
DATA_POINTS = 32

print("=== Experimental Configuration ===")
print(f"  Network:    {NUM_LAYERS}-layer deep linear, {DIM}x{DIM} weight matrices")
print(f"  Training:   {NUM_STEPS} steps, momentum={MOMENTUM}, {DATA_POINTS} data points")
print(f"  Seeds:      {NUM_SEEDS} independent runs")
print(f"  Rescaling:  gamma={GAMMA}, clamp=[{SCALE_MIN}, {SCALE_MAX}]")
print(f"  Predicted effective LR from clamping: 0.02 x {SCALE_MIN} = {0.02 * SCALE_MIN}")

In [ ]:
# LR sweep values
VANILLA_LRS = [0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1]
SGD_LRS = [0.0005, 0.001, 0.002, 0.005, 0.01, 0.02, 0.05, 0.1, 0.2, 0.5]
ORIGINAL_LR = 0.02  # the 3.4 default

print("=== Learning Rate Sweep Design ===")
print(f"  Vanilla Muon LRs: {VANILLA_LRS}")
print(f"  SGD LRs:          {SGD_LRS}")
print(f"  Original 3.4 LR:  {ORIGINAL_LR}")
print(f"")
print(f"  The sweep is log-spaced and brackets the predicted optimal LR (0.002).")
print(f"  We include the original LR (0.02) in the sweep for direct comparison.")
print(f"  SGD sweep extends to higher LRs (0.2, 0.5) since SGD can tolerate larger steps")
print(f"  in this architecture without the amplification effect of Newton-Schulz orthogonalization.")

## Network Utilities

### Deep Linear Network Primitives

These functions are identical to Experiment 3.4 to ensure exact reproducibility. The deep linear network
is defined by a sequence of weight matrices $W_1, W_2, \ldots, W_L$ with the forward pass:

$$Y = W_L \cdot W_{L-1} \cdots W_1 \cdot X$$

**Why deep linear networks?** Despite having no nonlinearities, deep linear nets exhibit rich optimization
dynamics: loss landscape saddle points, implicit regularization toward low-rank solutions, and layer-dependent
gradient scaling. They are the simplest setting where depth-related optimization phenomena (including the
effects we study here) manifest cleanly.

**Initialization**: Near-identity ($I + 0.1 \cdot \mathcal{N}(0,1)$) so the product $W_L \cdots W_1 \approx I$
at initialization. This ensures all methods start from a comparable, well-conditioned point.

In [ ]:
def init_weights(dim, num_layers, seed):
    """Initialize layers near identity."""
    rng = np.random.RandomState(seed)
    weights = []
    for _ in range(num_layers):
        W = np.eye(dim) + rng.randn(dim, dim) * 0.1
        weights.append(W.copy())
    return weights

In [ ]:
def forward_linear(weights, X):
    """Forward pass through deep linear net."""
    out = X.copy()
    for W in weights:
        out = W @ out
    return out

In [ ]:
def compute_loss(weights, X, Y_target):
    """MSE loss."""
    Y_pred = forward_linear(weights, X)
    diff = Y_pred - Y_target
    return 0.5 * np.mean(diff ** 2)

In [ ]:
def compute_gradients(weights, X, Y_target):
    """Backprop through deep linear net."""
    num_layers = len(weights)
    batch_size = X.shape[1]

    activations = [X.copy()]
    for W in weights:
        activations.append(W @ activations[-1])

    delta = (activations[-1] - Y_target) / batch_size

    grads = [None] * num_layers
    for l in range(num_layers - 1, -1, -1):
        grads[l] = delta @ activations[l].T
        if l > 0:
            delta = weights[l].T @ delta

    return grads

## Newton-Schulz Orthogonalization

### The Heart of Muon's Update Direction

Newton-Schulz iteration is Muon's core mechanism for computing an approximate orthogonal polar factor of the
gradient. Given gradient $G$, it computes $G_{\text{orth}} \approx U V^T$ where $G = U \Sigma V^T$ is the SVD.

**The quintic polynomial** (coefficients a=3.4445, b=-4.7750, c=2.0315):

$$X_{k+1} = a X_k + b X_k (X_k^T X_k) + c X_k (X_k^T X_k)^2$$

This converges to the nearest orthogonal matrix in ~5 iterations. The key effect: **all singular values
of the update direction are set to 1**, equalizing the step size across all spectral components of the
gradient. This is fundamentally different from SGD, where large singular values dominate the update.

**Relevance to this experiment**: The orthogonalization changes the *magnitude* of the gradient dramatically.
The ratio $\|G\|_F / \|G_{\text{orth}}\|_F$ measures how much the norm changes. The curvature rescaler
uses this ratio to "undo" some of the norm change, but in practice, the ratio is consistently small
(gradient norm << orthogonalized norm), so the rescaler clamps to its minimum. The question is whether
this clamping behavior constitutes genuine adaptation or just LR reduction.

In [ ]:
def newton_schulz_orthogonalize(G, num_iters=5):
    """Newton-Schulz iteration: Muon's quintic polynomial."""
    a, b, c = 3.4445, -4.7750, 2.0315
    norm = np.linalg.norm(G, 'fro')
    if norm < 1e-12:
        return G
    X = G / norm

    for _ in range(num_iters):
        XtX = X.T @ X
        X = a * X + b * (X @ XtX) + c * (X @ (XtX @ XtX))

    return X

## Optimizers

### Muon (with optional curvature rescaling) and SGD

Two optimizers are implemented for this comparison:

**Muon (train_muon)** operates in two modes:
- `rescale_mode='none'` (vanilla): Standard Muon -- orthogonalize gradient via Newton-Schulz, apply with
  momentum. The update magnitude is controlled solely by `lr`.
- `rescale_mode='curvature'`: After orthogonalization, multiply by
  $\text{scale} = \text{clip}(\|G\|_F / \|G_{\text{orth}}\|_F \cdot \gamma, \ \text{min}, \ \text{max})$.
  This attempts to restore the original gradient norm, but in practice almost always clamps to `scale_min=0.1`.

**SGD (train_sgd)**: Standard SGD with heavy-ball momentum. Serves as a non-orthogonalizing baseline to
contextualize the LR sensitivity: if SGD also needs a much lower LR than 0.02, the issue is architecture-specific,
not Muon-specific.

**Divergence guard**: Both optimizers detect loss > 1e10 or NaN and pad the remaining trajectory with `inf`.
This prevents wasted computation on diverged runs and makes the sweep robust to unstable LR choices.

In [ ]:
def train_muon(weights, X, Y_target, lr, num_steps, ns_iters=5,
               rescale_mode='none', gamma=1.0, scale_min=0.1, scale_max=10.0,
               momentum=0.9):
    """
    Muon optimizer with optional curvature rescaling.
    Returns (loss_history, scale_history, final_weights).
    """
    num_layers = len(weights)
    velocities = [np.zeros_like(W) for W in weights]
    losses = []
    scales_used = []

    for step in range(num_steps):
        loss = compute_loss(weights, X, Y_target)
        losses.append(loss)

        # Divergence guard
        if not np.isfinite(loss) or loss > 1e10:
            for remaining in range(num_steps - step):
                losses.append(float('inf'))
                scales_used.append(1.0)
            break

        grads = compute_gradients(weights, X, Y_target)

        step_scales = []
        for i in range(num_layers):
            G = grads[i]
            G_norm = np.linalg.norm(G, 'fro')

            G_orth = newton_schulz_orthogonalize(G, num_iters=ns_iters)
            G_orth_norm = np.linalg.norm(G_orth, 'fro')

            if rescale_mode == 'curvature':
                if G_orth_norm > 1e-12:
                    scale = np.clip(G_norm / G_orth_norm * gamma, scale_min, scale_max)
                else:
                    scale = 1.0
                G_orth = G_orth * scale
            else:
                scale = 1.0

            step_scales.append(scale)

            velocities[i] = momentum * velocities[i] + G_orth
            weights[i] = weights[i] - lr * velocities[i]

        scales_used.append(np.mean(step_scales))

    final_loss = compute_loss(weights, X, Y_target)
    losses.append(final_loss)

    return losses, scales_used, weights

In [ ]:
def train_sgd(weights, X, Y_target, lr, num_steps, momentum=0.9):
    """SGD with momentum."""
    num_layers = len(weights)
    velocities = [np.zeros_like(W) for W in weights]
    losses = []

    for step in range(num_steps):
        loss = compute_loss(weights, X, Y_target)
        losses.append(loss)

        if not np.isfinite(loss) or loss > 1e10:
            for remaining in range(num_steps - step):
                losses.append(float('inf'))
            break

        grads = compute_gradients(weights, X, Y_target)

        for i in range(num_layers):
            velocities[i] = momentum * velocities[i] + grads[i]
            weights[i] = weights[i] - lr * velocities[i]

    final_loss = compute_loss(weights, X, Y_target)
    losses.append(final_loss)

    return losses, weights

## Data Generation

### Synthetic Regression Targets

Each seed generates an independent regression problem: a random target network $W^*_1, W^*_2$ (entries
drawn from $\mathcal{N}(0, 0.3)$) and 32 input vectors $X \sim \mathcal{N}(0, 0.5)$. The target output
is $Y^* = W^*_2 W^*_1 X$.

The seed scheme uses `seed + 1000` for weight initialization to ensure the initial weights are independent
of the target, while remaining deterministic. This is identical to 3.4 so results are directly comparable.

In [ ]:
def make_problem(seed):
    """Generate target and data for a single seed."""
    rng = np.random.RandomState(seed)
    W_target = [rng.randn(DIM, DIM) * 0.3 for _ in range(NUM_LAYERS)]
    X = rng.randn(DIM, DATA_POINTS) * 0.5
    Y_target = X.copy()
    for W in W_target:
        Y_target = W @ Y_target
    return X, Y_target

## Main Experiment: Five-Phase Artifact Analysis

### Execution Flow

The experiment proceeds through five distinct phases, each building on the previous:

1. **Phase 1 -- Vanilla Muon LR Sweep**: Train vanilla Muon across 8 learning rates, identify the optimal
   LR. If this optimal LR is near 0.002 (= 0.02 x 0.1), the rescaler is likely just LR reduction.

2. **Phase 2 -- SGD LR Sweep**: Train SGD across 10 learning rates for baseline context. This tells us
   whether the LR sensitivity is Muon-specific or architecture-general.

3. **Phase 3 -- Curvature-Rescaled Muon**: Run rescaled Muon at both the original LR (0.02, replicating 3.4)
   and at the best vanilla LR. The comparison between these two reveals whether rescaling adds value
   beyond what LR tuning already provides.

4. **Phase 4 -- Comprehensive Table**: Side-by-side comparison across all methods and LRs.

5. **Phase 5 -- Hypothesis Tests**: Formal evaluation of T1, T2, T3 with quantitative pass/fail criteria.

In [ ]:
def main():
    seeds = [42 + i * 137 for i in range(NUM_SEEDS)]

    print()
    print("=" * 110)
    print("  Experiment H6: LR Artifact Check -- Is the 130x from curvature rescaling just LR reduction?")
    print("=" * 110)
    print()
    print(f"  Setup: {NUM_LAYERS}-layer {DIM}x{DIM} deep linear, {NUM_STEPS} steps, {NUM_SEEDS} seeds")
    print(f"  Original 3.4 used lr={ORIGINAL_LR} with curvature rescale hitting min-clamp (0.1) 96.5% of time")
    print(f"  Hypothesis: effective LR was ~{ORIGINAL_LR * 0.1} => sweeping vanilla Muon LR to check")
    print()
    print(f"  Seeds used: {seeds}")
    print(f"  Weight init seeds: {[s + 1000 for s in seeds]}")
    print()

    # =========================================================================
    # PHASE 1: Vanilla Muon LR sweep
    # =========================================================================
    print("-" * 110)
    print("  PHASE 1: Vanilla Muon LR sweep")
    print("-" * 110)
    print()
    print("  Rationale: If the curvature rescaler is just multiplying by 0.1 (its min-clamp)")
    print("  on 96.5% of steps, then vanilla Muon at lr=0.002 should match rescaled Muon at lr=0.02.")
    print("  We sweep 8 LRs spanning 200x range to find the true optimum.")
    print()

    vanilla_results = {}  # lr -> list of final losses across seeds
    for lr in VANILLA_LRS:
        final_losses = []
        for seed in seeds:
            X, Y_target = make_problem(seed)
            w_init = init_weights(DIM, NUM_LAYERS, seed + 1000)
            losses, _, _ = train_muon(w_init, X, Y_target, lr=lr, num_steps=NUM_STEPS,
                                       ns_iters=5, rescale_mode='none', momentum=MOMENTUM)
            final_losses.append(losses[-1])
        vanilla_results[lr] = final_losses
        mean_l = np.mean(final_losses)
        med_l = np.median(final_losses)
        finite_frac = np.mean(np.isfinite(final_losses)) * 100
        # Flag diverged or suspicious runs
        n_diverged = sum(1 for fl in final_losses if not np.isfinite(fl) or fl > 1e6)
        status = ""
        if n_diverged > 0:
            status = f"  ** {n_diverged}/{NUM_SEEDS} seeds diverged **"
        elif mean_l < 1e-6:
            status = "  (excellent convergence)"
        print(f"    lr={lr:<8.4f}  mean={mean_l:12.6e}  median={med_l:12.6e}  "
              f"finite={finite_frac:.0f}%{status}")

    # Find best vanilla LR (by mean of finite losses)
    best_vanilla_lr = None
    best_vanilla_mean = float('inf')
    for lr in VANILLA_LRS:
        fl = np.array(vanilla_results[lr])
        finite_fl = fl[np.isfinite(fl)]
        if len(finite_fl) > 0:
            m = np.mean(finite_fl)
            if m < best_vanilla_mean:
                best_vanilla_mean = m
                best_vanilla_lr = lr

    print()
    print(f"    >>> Best vanilla Muon LR: {best_vanilla_lr} (mean loss = {best_vanilla_mean:.6e})")
    print(f"    >>> Expected if rescaler is pure LR reduction: ~{ORIGINAL_LR * 0.1}")
    print(f"    >>> Ratio best/expected: {best_vanilla_lr / (ORIGINAL_LR * 0.1):.2f}x")
    print()
    if abs(best_vanilla_lr - ORIGINAL_LR * 0.1) < 1e-10:
        print("    ** EXACT MATCH: Best vanilla LR is precisely the predicted effective LR from clamping! **")
    elif 0.5 <= best_vanilla_lr / (ORIGINAL_LR * 0.1) <= 2.0:
        print("    ** APPROXIMATE MATCH: Best vanilla LR is within 2x of the predicted effective LR. **")
    else:
        print("    ** MISMATCH: Best vanilla LR does NOT match the predicted effective LR from clamping. **")
    print()

    # Print the loss landscape shape around the optimum
    print("    Loss landscape around optimum (log10 of mean loss):")
    for lr in VANILLA_LRS:
        fl = np.array(vanilla_results[lr])
        finite_fl = fl[np.isfinite(fl)]
        if len(finite_fl) > 0:
            log_loss = np.log10(np.mean(finite_fl))
            bar = "#" * max(1, int(50 + 5 * log_loss))  # rough visual bar
            marker = " <-- BEST" if abs(lr - best_vanilla_lr) < 1e-10 else ""
            marker += " <-- 3.4 default" if abs(lr - ORIGINAL_LR) < 1e-10 else ""
            print(f"      lr={lr:<8.4f}  log10(loss)={log_loss:+6.2f}  {bar}{marker}")
        else:
            print(f"      lr={lr:<8.4f}  DIVERGED")
    print()

    # =========================================================================
    # PHASE 2: SGD LR sweep
    # =========================================================================
    print("-" * 110)
    print("  PHASE 2: SGD LR sweep")
    print("-" * 110)
    print()
    print("  Rationale: SGD serves as a non-orthogonalizing baseline. If SGD's best LR is also")
    print("  far below 0.02, the problem is architecture-wide, not Muon-specific. If SGD thrives")
    print("  at 0.02 but Muon doesn't, the issue is specific to how Newton-Schulz amplifies updates.")
    print()

    sgd_results = {}
    for lr in SGD_LRS:
        final_losses = []
        for seed in seeds:
            X, Y_target = make_problem(seed)
            w_init = init_weights(DIM, NUM_LAYERS, seed + 1000)
            losses, _ = train_sgd(w_init, X, Y_target, lr=lr, num_steps=NUM_STEPS,
                                   momentum=MOMENTUM)
            final_losses.append(losses[-1])
        sgd_results[lr] = final_losses
        mean_l = np.mean(final_losses)
        finite_frac = np.mean(np.isfinite(final_losses)) * 100
        n_diverged = sum(1 for fl in final_losses if not np.isfinite(fl) or fl > 1e6)
        status = ""
        if n_diverged > 0:
            status = f"  ** {n_diverged}/{NUM_SEEDS} seeds diverged **"
        print(f"    lr={lr:<8.4f}  mean={mean_l:12.6e}  finite={finite_frac:.0f}%{status}")

    best_sgd_lr = None
    best_sgd_mean = float('inf')
    for lr in SGD_LRS:
        fl = np.array(sgd_results[lr])
        finite_fl = fl[np.isfinite(fl)]
        if len(finite_fl) > 0:
            m = np.mean(finite_fl)
            if m < best_sgd_mean:
                best_sgd_mean = m
                best_sgd_lr = lr

    print(f"\n    >>> Best SGD LR: {best_sgd_lr} (mean loss = {best_sgd_mean:.6e})")
    print()
    print(f"    Comparison: Muon best LR = {best_vanilla_lr}, SGD best LR = {best_sgd_lr}")
    print(f"    Ratio (SGD best / Muon best): {best_sgd_lr / best_vanilla_lr:.1f}x")
    if best_sgd_lr > best_vanilla_lr * 5:
        print("    >> SGD tolerates much higher LR than Muon, suggesting Newton-Schulz amplifies updates.")
    elif best_sgd_lr < best_vanilla_lr * 0.2:
        print("    >> SGD needs even lower LR, suggesting the problem is architecture-wide.")
    else:
        print("    >> Similar LR ranges for SGD and Muon.")
    print()

    # =========================================================================
    # PHASE 3: Curvature-rescaled Muon at lr=0.02 (3.4 reference)
    # =========================================================================
    print("-" * 110)
    print("  PHASE 3: Curvature-rescaled Muon")
    print("-" * 110)
    print()
    print("  Phase 3a: Replicate 3.4 -- rescaled Muon at lr=0.02 (the original setting)")
    print("  Phase 3b: Rescaled Muon at the best vanilla LR -- does rescaling add further value?")
    print()

    # 3a: Rescaled at original LR (the 3.4 result)
    print("  --- Phase 3a: Rescaled Muon at lr=0.02 (3.4 replication) ---")
    rescaled_orig_losses = []
    rescaled_orig_scales = []
    for seed in seeds:
        X, Y_target = make_problem(seed)
        w_init = init_weights(DIM, NUM_LAYERS, seed + 1000)
        losses, scales, _ = train_muon(w_init, X, Y_target, lr=ORIGINAL_LR,
                                        num_steps=NUM_STEPS, ns_iters=5,
                                        rescale_mode='curvature', gamma=GAMMA,
                                        scale_min=SCALE_MIN, scale_max=SCALE_MAX,
                                        momentum=MOMENTUM)
        rescaled_orig_losses.append(losses[-1])
        rescaled_orig_scales.extend(scales)

    rescaled_orig_mean = np.mean(rescaled_orig_losses)
    scales_arr = np.array(rescaled_orig_scales)
    hit_min_pct = np.mean(scales_arr <= SCALE_MIN + 1e-8) * 100

    print(f"    Rescaled Muon at lr={ORIGINAL_LR}:  mean loss = {rescaled_orig_mean:.6e}")
    print(f"    Scale factor stats: mean={np.mean(scales_arr):.4f}, "
          f"median={np.median(scales_arr):.4f}, "
          f"min={np.min(scales_arr):.4f}, max={np.max(scales_arr):.4f}")
    print(f"    Fraction hitting min clamp ({SCALE_MIN}): {hit_min_pct:.1f}%")
    print()

    # Detailed scale distribution
    print(f"    Scale factor distribution (all {len(scales_arr)} measurements across seeds and steps):")
    for threshold in [0.1, 0.2, 0.5, 1.0, 2.0, 5.0, 10.0]:
        frac_below = np.mean(scales_arr <= threshold + 1e-8) * 100
        print(f"      scale <= {threshold:<5.1f}: {frac_below:5.1f}%")
    print()

    if hit_min_pct > 90:
        print(f"    ** CONFIRMED: Scale factor is clamped to min ({SCALE_MIN}) in {hit_min_pct:.1f}% of steps.")
        print(f"    ** This means the rescaler is effectively just multiplying by {SCALE_MIN} = reducing LR by {1/SCALE_MIN:.0f}x.")
        print(f"    ** Effective LR: {ORIGINAL_LR} x {SCALE_MIN} = {ORIGINAL_LR * SCALE_MIN}")
    print()

    # 3b: Rescaled at best vanilla LR
    print(f"  --- Phase 3b: Rescaled Muon at lr={best_vanilla_lr} (best vanilla LR) ---")
    print(f"  This is the critical test: if rescaling provides genuine curvature adaptation,")
    print(f"  it should FURTHER improve upon the already-optimal vanilla LR.")
    print()
    rescaled_best_losses = []
    rescaled_best_scales = []
    for seed in seeds:
        X, Y_target = make_problem(seed)
        w_init = init_weights(DIM, NUM_LAYERS, seed + 1000)
        losses, scales, _ = train_muon(w_init, X, Y_target, lr=best_vanilla_lr,
                                        num_steps=NUM_STEPS, ns_iters=5,
                                        rescale_mode='curvature', gamma=GAMMA,
                                        scale_min=SCALE_MIN, scale_max=SCALE_MAX,
                                        momentum=MOMENTUM)
        rescaled_best_losses.append(losses[-1])
        rescaled_best_scales.extend(scales)

    rescaled_best_mean = np.mean(rescaled_best_losses)
    scales_best_arr = np.array(rescaled_best_scales)
    hit_min_best_pct = np.mean(scales_best_arr <= SCALE_MIN + 1e-8) * 100

    print(f"    Rescaled Muon at lr={best_vanilla_lr} (best vanilla LR):  mean loss = {rescaled_best_mean:.6e}")
    print(f"    Scale factor stats: mean={np.mean(scales_best_arr):.4f}, "
          f"median={np.median(scales_best_arr):.4f}, "
          f"min={np.min(scales_best_arr):.4f}, max={np.max(scales_best_arr):.4f}")
    print(f"    Fraction hitting min clamp ({SCALE_MIN}): {hit_min_best_pct:.1f}%")
    print()

    # Compare scale distributions between 3a and 3b
    print(f"    Scale distribution comparison:")
    print(f"      At lr=0.02:             mean_scale={np.mean(scales_arr):.4f}, min_clamp_frac={hit_min_pct:.1f}%")
    print(f"      At lr={best_vanilla_lr} (best): mean_scale={np.mean(scales_best_arr):.4f}, min_clamp_frac={hit_min_best_pct:.1f}%")
    if hit_min_best_pct > hit_min_pct:
        print(f"    ** At the lower LR, even MORE steps hit the min clamp -- rescaling is even less useful. **")
    elif hit_min_best_pct < hit_min_pct - 10:
        print(f"    ** At the lower LR, FEWER steps hit the min clamp -- rescaling may be doing something adaptive. **")
    print()

    # =========================================================================
    # PHASE 4: COMPREHENSIVE TABLE
    # =========================================================================
    print("=" * 110)
    print("  COMPREHENSIVE RESULTS TABLE")
    print("=" * 110)
    print()
    print("  This table shows all methods side-by-side. The key comparison is between:")
    print("  - Vanilla Muon at the best LR (found in Phase 1)")
    print("  - Rescaled Muon at lr=0.02 (the 3.4 reference)")
    print("  If these match, the 130x is an LR artifact.")
    print()

    # Collect all Muon results into table rows
    print(f"  {'LR':>8}  |  {'Vanilla Muon':>16}  {'(std)':>12}  |  "
          f"{'Rescaled Muon':>16}  {'(std)':>12}  |  {'SGD':>16}  {'(std)':>12}")
    print(f"  {'':->8}--+--{'':->16}--{'':->12}--+--"
          f"{'':->16}--{'':->12}--+--{'':->16}--{'':->12}")

    for lr in VANILLA_LRS:
        # Vanilla Muon
        vm = np.array(vanilla_results[lr])
        vm_finite = vm[np.isfinite(vm)]
        vm_str = f"{np.mean(vm_finite):16.6e}" if len(vm_finite) > 0 else f"{'DIVERGED':>16}"
        vm_std = f"{np.std(vm_finite):12.2e}" if len(vm_finite) > 0 else f"{'---':>12}"

        # Rescaled Muon (only computed for original and best)
        if abs(lr - ORIGINAL_LR) < 1e-10:
            rm = np.array(rescaled_orig_losses)
            rm_str = f"{np.mean(rm):16.6e}"
            rm_std = f"{np.std(rm):12.2e}"
        elif abs(lr - best_vanilla_lr) < 1e-10 and abs(best_vanilla_lr - ORIGINAL_LR) > 1e-10:
            rm = np.array(rescaled_best_losses)
            rm_str = f"{np.mean(rm):16.6e}"
            rm_std = f"{np.std(rm):12.2e}"
        else:
            rm_str = f"{'---':>16}"
            rm_std = f"{'---':>12}"

        # SGD
        if lr in sgd_results:
            sg = np.array(sgd_results[lr])
            sg_finite = sg[np.isfinite(sg)]
            sg_str = f"{np.mean(sg_finite):16.6e}" if len(sg_finite) > 0 else f"{'DIVERGED':>16}"
            sg_std = f"{np.std(sg_finite):12.2e}" if len(sg_finite) > 0 else f"{'---':>12}"
        else:
            sg_str = f"{'---':>16}"
            sg_std = f"{'---':>12}"

        marker = ""
        if abs(lr - ORIGINAL_LR) < 1e-10:
            marker = "  <-- 3.4 default"
        if abs(lr - best_vanilla_lr) < 1e-10:
            marker += "  <-- BEST vanilla"

        print(f"  {lr:>8.4f}  |  {vm_str}  {vm_std}  |  "
              f"{rm_str}  {rm_std}  |  {sg_str}  {sg_std}{marker}")

    print()

    # Also print the rescaled result at best vanilla LR if not already in table
    if best_vanilla_lr not in VANILLA_LRS:
        print(f"  NOTE: best_vanilla_lr={best_vanilla_lr} not in sweep -- this should not happen.")

    # =========================================================================
    # PHASE 5: KEY HYPOTHESIS TESTS
    # =========================================================================
    print("=" * 110)
    print("  KEY HYPOTHESIS TESTS")
    print("=" * 110)
    print()
    print("  These are the three formal tests that determine whether the 130x is an artifact.")
    print("  Each test has a clear pass/fail criterion defined before the experiment.")
    print()

    # Reference values
    vanilla_orig = np.array(vanilla_results[ORIGINAL_LR])
    vanilla_orig_mean = np.mean(vanilla_orig[np.isfinite(vanilla_orig)]) if np.any(np.isfinite(vanilla_orig)) else float('inf')
    vanilla_best = np.array(vanilla_results[best_vanilla_lr])
    vanilla_best_mean = np.mean(vanilla_best[np.isfinite(vanilla_best)])

    print(f"  Reference values:")
    print(f"    Vanilla Muon at lr={ORIGINAL_LR} (3.4 default):    mean = {vanilla_orig_mean:.6e}")
    print(f"    Vanilla Muon at lr={best_vanilla_lr} (best):         mean = {vanilla_best_mean:.6e}")
    print(f"    Rescaled Muon at lr={ORIGINAL_LR} (3.4 result):    mean = {rescaled_orig_mean:.6e}")
    print(f"    Rescaled Muon at lr={best_vanilla_lr} (best+resc):   mean = {rescaled_best_mean:.6e}")
    print(f"    Best SGD at lr={best_sgd_lr}:                       mean = {best_sgd_mean:.6e}")
    print()

    # Improvement ratios
    print("  Improvement ratios (higher = more improvement from changing method/LR):")
    if rescaled_orig_mean > 1e-15:
        ratio_orig_vs_rescaled = vanilla_orig_mean / rescaled_orig_mean
        print(f"    vanilla(0.02) / rescaled(0.02) = {ratio_orig_vs_rescaled:.1f}x")
        print(f"      (This is the '130x' improvement from 3.4)")
    print()

    if vanilla_best_mean > 1e-15:
        ratio_best_vs_orig = vanilla_orig_mean / vanilla_best_mean
        print(f"    vanilla(0.02) / vanilla(best={best_vanilla_lr}) = {ratio_best_vs_orig:.1f}x")
        print(f"      (How much improvement comes from LR tuning alone -- no rescaling needed)")
    print()

    if vanilla_best_mean > 1e-15 and rescaled_orig_mean > 1e-15:
        print(f"    If vanilla(best) / rescaled(0.02) is near 1.0, the 130x is purely an LR artifact.")
        print(f"    Actual ratio: {vanilla_best_mean / rescaled_orig_mean:.4f}")
    print()

    # --- T1: Is the best vanilla LR near 0.002? ---
    print("  " + "-" * 106)
    expected_lr = ORIGINAL_LR * SCALE_MIN  # 0.02 * 0.1 = 0.002
    t1_ratio = best_vanilla_lr / expected_lr
    t1_pass = 0.5 <= t1_ratio <= 2.0  # within factor of 2
    t1_exact = abs(best_vanilla_lr - expected_lr) < 1e-10

    print(f"  T1: Is the best vanilla Muon LR near {expected_lr} (= {ORIGINAL_LR} x {SCALE_MIN})?")
    print(f"      Best vanilla LR = {best_vanilla_lr}")
    print(f"      Expected LR     = {expected_lr}")
    print(f"      Ratio best/expected = {t1_ratio:.2f}")
    print()
    print(f"      Pass criterion: ratio in [0.5, 2.0] => rescaler is primarily LR reduction")
    if t1_exact:
        print(f"      RESULT: EXACT MATCH -- best vanilla LR IS exactly the clamped LR")
    elif t1_pass:
        print(f"      RESULT: APPROXIMATE MATCH -- within factor of 2")
    else:
        print(f"      RESULT: NO MATCH -- best vanilla LR is far from expected")

    if t1_pass:
        print(f"      INTERPRETATION: The rescaler IS primarily acting as LR reduction.")
        print(f"      The min-clamp at {SCALE_MIN} effectively sets lr_eff = {ORIGINAL_LR * SCALE_MIN},")
        print(f"      and the optimizer just happens to need that lower LR.")
    else:
        print(f"      INTERPRETATION: The rescaler does something beyond simple LR reduction.")
        print(f"      The optimal LR is at {best_vanilla_lr}, not at the predicted {expected_lr}.")
    print()

    # --- T2: Does best-LR vanilla Muon match rescaled Muon within 5%? ---
    print("  " + "-" * 106)
    if rescaled_orig_mean > 1e-15:
        t2_ratio = vanilla_best_mean / rescaled_orig_mean
        t2_pct_diff = abs(t2_ratio - 1.0) * 100
        t2_pass = t2_pct_diff < 5.0
    else:
        t2_ratio = float('inf')
        t2_pct_diff = float('inf')
        t2_pass = False

    print(f"  T2: Does best-LR vanilla Muon match rescaled Muon at lr=0.02 within 5%?")
    print(f"      Vanilla best (lr={best_vanilla_lr}):  {vanilla_best_mean:.6e}")
    print(f"      Rescaled (lr={ORIGINAL_LR}):          {rescaled_orig_mean:.6e}")
    print(f"      Ratio = {t2_ratio:.4f}  (diff = {t2_pct_diff:.1f}%)")
    print()
    print(f"      Pass criterion: |ratio - 1.0| < 5% => the 130x is a pure LR artifact")
    if t2_pass:
        print(f"      RESULT: MATCH within 5% -- the 130x IS an artifact of bad default LR")
        print(f"      Simply reducing the LR by {1/SCALE_MIN:.0f}x achieves the same result as 'curvature rescaling'.")
    else:
        if vanilla_best_mean < rescaled_orig_mean:
            print(f"      RESULT: NO MATCH -- vanilla at optimal LR is BETTER than rescaled.")
            print(f"      The rescaling provides NO value; proper LR tuning is sufficient.")
            print(f"      In fact, the rescaler was sub-optimal -- it over-reduced the LR or")
            print(f"      introduced noise through the residual non-clamped steps.")
        else:
            print(f"      RESULT: NO MATCH -- rescaled Muon is still better than best vanilla.")
            print(f"      The rescaling provides genuine value beyond LR reduction.")
            print(f"      The 3.5% of non-clamped steps may carry important adaptive information.")
    print()

    # --- T3: Does rescaled Muon at best vanilla LR further improve? ---
    print("  " + "-" * 106)
    if vanilla_best_mean > 1e-15:
        t3_ratio = vanilla_best_mean / rescaled_best_mean
        t3_pct_improvement = (1.0 - rescaled_best_mean / vanilla_best_mean) * 100
    else:
        t3_ratio = float('nan')
        t3_pct_improvement = float('nan')
    t3_pass = rescaled_best_mean < vanilla_best_mean * 0.95  # >5% improvement

    print(f"  T3: Does rescaled Muon at the best vanilla LR further improve?")
    print(f"      Vanilla at lr={best_vanilla_lr}:            {vanilla_best_mean:.6e}")
    print(f"      Rescaled at lr={best_vanilla_lr}:           {rescaled_best_mean:.6e}")
    print(f"      Improvement from rescaling: {t3_pct_improvement:.1f}%  (ratio = {t3_ratio:.2f}x)")
    print()
    print(f"      Pass criterion: rescaled_loss < vanilla_loss x 0.95 => rescaling has genuine value")
    if t3_pass:
        print(f"      RESULT: YES -- rescaling provides >{5}% further improvement")
        print(f"      Curvature rescaling has genuine value beyond LR tuning.")
        print(f"      Even at the optimal LR, the per-step scale adaptation helps.")
    else:
        if rescaled_best_mean > vanilla_best_mean:
            print(f"      RESULT: NO -- rescaling actually HURTS at optimal LR")
            print(f"      Curvature rescaling is purely an LR artifact.")
            print(f"      At the optimal LR, the additional scale factor pushes the effective LR")
            print(f"      even lower (by another {SCALE_MIN}x = {best_vanilla_lr * SCALE_MIN}), making it too small.")
        else:
            print(f"      RESULT: MARGINAL -- rescaling helps <5%, essentially just LR effect")
            print(f"      The improvement ({t3_pct_improvement:.1f}%) is within noise.")
    print()

    # =========================================================================
    # PER-SEED COMPARISON: best vanilla vs rescaled
    # =========================================================================
    print("=" * 110)
    print("  PER-SEED COMPARISON: Vanilla(best LR) vs Rescaled(original LR) vs Rescaled(best LR)")
    print("=" * 110)
    print()
    print("  This per-seed breakdown reveals whether the aggregate statistics hide seed-specific effects.")
    print("  If one method consistently wins across all seeds, the result is robust.")
    print("  If wins are split, the difference may be within noise.")
    print()

    vanilla_best_arr = np.array(vanilla_results[best_vanilla_lr])
    rescaled_orig_arr = np.array(rescaled_orig_losses)
    rescaled_best_arr = np.array(rescaled_best_losses)

    print(f"  {'Seed':>6}  {'Vanilla(best)':>16}  {'Rescaled(0.02)':>16}  "
          f"{'Rescaled(best)':>16}  {'Winner':>20}")
    print(f"  {'':->6}  {'':->16}  {'':->16}  {'':->16}  {'':->20}")

    wins_vanilla = 0
    wins_resc_orig = 0
    wins_resc_best = 0
    for i in range(NUM_SEEDS):
        vb = vanilla_best_arr[i]
        ro = rescaled_orig_arr[i]
        rb = rescaled_best_arr[i]

        candidates = {'Vanilla(best)': vb, 'Resc(0.02)': ro, 'Resc(best)': rb}
        winner = min(candidates, key=lambda k: candidates[k] if np.isfinite(candidates[k]) else float('inf'))
        if winner == 'Vanilla(best)':
            wins_vanilla += 1
        elif winner == 'Resc(0.02)':
            wins_resc_orig += 1
        else:
            wins_resc_best += 1

        print(f"  {i+1:>6}  {vb:16.6e}  {ro:16.6e}  {rb:16.6e}  {winner:>20}")

    print()
    print(f"  Win counts: Vanilla(best)={wins_vanilla}, "
          f"Resc(0.02)={wins_resc_orig}, Resc(best)={wins_resc_best}")
    print()
    total_seeds = wins_vanilla + wins_resc_orig + wins_resc_best
    dominant_method = max([('Vanilla(best)', wins_vanilla), ('Resc(0.02)', wins_resc_orig),
                           ('Resc(best)', wins_resc_best)], key=lambda x: x[1])
    print(f"  Dominant method: {dominant_method[0]} ({dominant_method[1]}/{total_seeds} seeds)")
    if dominant_method[1] >= 8:
        print(f"  ** Strong dominance ({dominant_method[1]}/10 seeds) -- result is robust. **")
    elif dominant_method[1] >= 6:
        print(f"  ** Moderate dominance -- result is suggestive but not conclusive. **")
    else:
        print(f"  ** No clear winner -- methods are essentially tied at this resolution. **")
    print()

    # =========================================================================
    # OVERALL VERDICT
    # =========================================================================
    print("=" * 110)
    print("  OVERALL VERDICT")
    print("=" * 110)
    print()
    print(f"  Summary of hypothesis tests:")
    print(f"    T1 (best LR matches predicted):     {'PASS' if t1_pass else 'FAIL'}  (best={best_vanilla_lr}, predicted={expected_lr}, ratio={t1_ratio:.2f})")
    print(f"    T2 (vanilla matches rescaled):       {'PASS' if t2_pass else 'FAIL'}  (diff={t2_pct_diff:.1f}%)")
    print(f"    T3 (rescaling adds value at best LR): {'PASS' if t3_pass else 'FAIL'}  (improvement={t3_pct_improvement:.1f}%)")
    print()

    # Determine the story
    if t1_pass and t2_pass and not t3_pass:
        print("  VERDICT: THE 130x IS AN LR ARTIFACT.")
        print()
        print("  Evidence:")
        print(f"    1. Best vanilla LR ({best_vanilla_lr}) matches predicted LR "
              f"({expected_lr}) from clamp analysis.")
        print(f"    2. Vanilla at optimal LR matches rescaled Muon within {t2_pct_diff:.1f}%.")
        print(f"    3. Adding rescaling at optimal LR provides only {t3_pct_improvement:.1f}% change.")
        print()
        print("  The curvature rescaler in 3.4 accidentally found a better LR by")
        print("  multiplying by 0.1 (its min clamp) 96.5% of the time. The 'curvature")
        print("  adaptation' is a 10x LR reduction wearing a lab coat.")
    elif t3_pass:
        print("  VERDICT: CURVATURE RESCALING HAS GENUINE VALUE BEYOND LR TUNING.")
        print()
        print("  Evidence:")
        print(f"    1. Best vanilla LR: {best_vanilla_lr} (predicted: {expected_lr})")
        print(f"    2. Vanilla at optimal LR: {vanilla_best_mean:.6e}")
        print(f"    3. Rescaled at optimal LR: {rescaled_best_mean:.6e} ({t3_pct_improvement:.1f}% better)")
        print()
        print("  Even after correcting the LR, rescaling provides further improvement.")
        partially = ""
        if t1_pass:
            partially = "PARTIALLY an LR artifact (most of the 130x is LR), but "
        print(f"  The 130x is {partially}rescaling adds genuine per-step adaptation.")
    else:
        print("  VERDICT: MIXED / COMPLEX RESULT.")
        print()
        print(f"    T1 (best LR near 0.002): {'PASS' if t1_pass else 'FAIL'} -- best LR = {best_vanilla_lr}")
        print(f"    T2 (vanilla matches rescaled): {'PASS' if t2_pass else 'FAIL'} -- diff = {t2_pct_diff:.1f}%")
        print(f"    T3 (rescaling adds value at best LR): {'PASS' if t3_pass else 'FAIL'} -- {t3_pct_improvement:.1f}%")
        print()
        if not t1_pass and not t2_pass:
            print("  The rescaler is NOT primarily acting as LR reduction. The mechanism")
            print("  appears to be something more subtle than a simple constant multiplier.")
        elif t1_pass and not t2_pass:
            print("  The LR is in the right ballpark but the match is not within 5%.")
            print("  The rescaler is mostly LR reduction but with some additional effect.")

    print()
    print("=" * 110)
    print("  EXPERIMENT COMPLETE")
    print("=" * 110)
    print()

## Execute Experiment

Running the full five-phase analysis. The output below will report:
- Phase 1: Vanilla Muon LR sweep with per-LR diagnostics and a text-based loss landscape visualization
- Phase 2: SGD LR sweep with cross-optimizer LR sensitivity comparison
- Phase 3: Curvature-rescaled Muon with detailed scale factor distributions at both LRs
- Phase 4: Comprehensive side-by-side results table
- Phase 5: Formal hypothesis tests T1/T2/T3 with per-seed robustness analysis and final verdict

In [ ]:
if __name__ == '__main__':
    main()

## Conclusions and Implications

### What This Experiment Reveals

This experiment applies the most basic scientific control to the 130x curvature advantage claim: **did we
just have the wrong learning rate?** The curvature rescaler in Experiment 3.4 was hitting its minimum clamp
(0.1) on 96.5% of steps, which is algebraically equivalent to a 10x LR reduction.

### Interpreting the Three Tests

- **T1 (LR Prediction)**: If the best vanilla LR is near 0.002 = 0.02 x 0.1, it confirms the rescaler's
  dominant effect is a constant LR multiplier, not adaptive curvature tracking.

- **T2 (Loss Matching)**: If properly-tuned vanilla Muon matches rescaled Muon within 5%, the entire 130x
  claim collapses to "Muon's default LR was 10x too high for this architecture."

- **T3 (Residual Value)**: Even if T1 and T2 pass (confirming the artifact), T3 asks whether rescaling
  provides *any* additional benefit at the correct LR. This separates "mostly artifact" from "entirely artifact."

### Implications for the Muon-as-Gauge-Fixing Theory

If the 130x is confirmed as an LR artifact:
- The curvature rescaling mechanism needs fundamental redesign (wider clamp range, adaptive gamma, etc.)
- The *direction* provided by Newton-Schulz may still be valuable, but the *scale* calibration was wrong
- Future experiments must always include LR-tuned baselines before claiming optimizer advantages

If rescaling shows genuine residual value (T3 passes):
- The 3.5% of non-clamped steps may carry disproportionate importance (e.g., near saddle points)
- The rescaler has signal, but it is masked by the dominant clamping behavior
- A redesigned rescaler with a tighter clamp range could amplify the adaptive component